# Cross-study gene expression comparison

Compare expression of a target gene (and optional housekeepers) across all datasets.
Uses `get_gene_stats_all()` — a single API call that queries all databases with a
`default_layer` server-side. Gene names are matched case-insensitively by default,
so `APOE` will match `Apoe` in mouse datasets.

In [ ]:
try:
    import micropip
    # 'vark' is vendored next to the notebook (top-level package).
    await micropip.install(['pandas', 'matplotlib'])
except:
    pass  # already installed in standard Python
print('ready')

In [ ]:
from vark import AnthiveClient
import pandas as pd
import matplotlib.pyplot as plt

try:
    import js
    api_url = f"{js.location.origin}/api/"
except:
    api_url = None

client = AnthiveClient(api_url)
print(f'Connected to {client.base_url}')

## Configuration

`target_gene` is the gene you're interested in. `housekeepers` are reference genes
used to normalise or provide context (set to `[]` to skip).
Gene names are matched case-insensitively — `APOE` will find `Apoe` in mouse datasets.

In [ ]:
target_gene  = 'Apoe'
housekeepers = ['Actb', 'Gapdh']  # set [] to skip

## Collect statistics across studies

A single API call queries all databases with a `default_layer` — no manual loop,
no large data transfer. Databases without a `default_layer` are skipped server-side.

In [ ]:
genes_to_query = [target_gene] + housekeepers

df = client.get_gene_stats_all(genes_to_query, case_sensitive=False, format='dataframe')

print(f'{len(df)} rows across {df["db_id"].nunique()} databases')
df.head()

## Plot: mean expression and detection rate

Left panel: mean expression ± std across all cells (zeros included).
Right panel: % cells expressing the gene.

In [ ]:
# Filter to target gene; drop rows where gene was not found in a database
tgt = df[df['gene'] == target_gene].copy()
if 'error' in tgt.columns:
    tgt = tgt[tgt['error'].isna()]
tgt = tgt.sort_values('mean', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(13, max(3, len(tgt) * 0.38)))

# Left: mean ± std (all cells, zeros included)
ax = axes[0]
ax.barh(tgt['title'], tgt['mean'], xerr=tgt['std'],
        color='steelblue', alpha=0.75, capsize=3, ecolor='gray')
ax.set_xlabel('Mean expression (all cells)')
ax.set_title(f'{target_gene} mean expression')

# Right: detection rate
ax = axes[1]
ax.barh(tgt['title'], tgt['pct_nonzero'], color='tomato', alpha=0.75)
ax.set_xlabel('% expressing cells')
ax.set_title(f'{target_gene} detection rate')

plt.tight_layout()
plt.show()

## Housekeepers: normalised expression

Express the target gene relative to each housekeeper's mean (ratio). This helps correct for differences in sequencing depth or normalisation across studies.

In [ ]:
if not housekeepers:
    print('No housekeepers configured — skip this cell')
else:
    # Pivot to wide format: one row per database
    wide = df.pivot_table(index='db_id', columns='gene', values='mean', aggfunc='first').reset_index()
    if target_gene not in wide.columns:
        print(f'{target_gene} not found in any database')
    else:
        fig, axes = plt.subplots(1, len(housekeepers), figsize=(5 * len(housekeepers), 4), squeeze=False)
        for i, hk in enumerate(housekeepers):
            ax = axes[0][i]
            if hk not in wide.columns:
                ax.set_title(f'{hk} — not found')
                continue
            ratio = wide[target_gene] / wide[hk].replace(0, float('nan'))
            order = ratio.sort_values(ascending=True)
            ax.barh(wide.loc[order.index, 'db_id'], order.values, color='mediumpurple', alpha=0.75)
            ax.set_xlabel(f'{target_gene} / {hk}')
            ax.set_title(f'Relative to {hk}')
        plt.tight_layout()
        plt.show()